## **CMIP Climate Stripes with Julia**

In [ ]:
print("Hello World")

#### **Python vs Julia**

**Design**
- Both Python and Julia are high level languages (human readible)
- Both languages use dynamic typing, so variables types do not need to be defined beforehand
- However, they differ in the way how code is translated to machine code
    - **Python** is an interpreter languge and compiles code into intermediate bytecode (saved in `__pycache__`), which the interpreter then executes step-by-step at runtime without compiling it into native CPU machine code.
    - **Julia** uses JIT (just in time) compilation, where a function is compiled just before it is called based on the type of the arguments that are passed to the function (multiple dispatch)
    - In **C** the whole program code is compiled before execution

**Syntax**
- Julia has 1-based indexing (vs. Python 0 based), e.g. `x=[0,1,2], x[1]=0`
- Julia needs `end` keyword for block termination but does not care about indentation 
- Julia differentiates between characters (single quotes: `'h'`) and strings (double quotes: `"hello world"`)
- Macros are statements that start with an `@` through which we can give commands to the compiler

**Speed**
- Python has functions that are precompiled in **C**
    - If you use only these functions, python will not be slower than Julia
    - If you use a lot of custom functions with loops and conditionals, they will be faster in Julia
- Generally
    - Use python for deep learning, tabular data processing (pandas) or general software engineering (object oriented)
    - Use julia for scientific coding (differential equations, ABMs, physical models, ...) and mathematics and optimisation

In the following we are going to showcase these performance differences between Python and Julia based on the CMIP data. 

### **Example**: Distance Calculation
Calculate the eucledian distance on two vectors with 10 Mio elements ($d=\sqrt{\sum(a_i-b_i)^2}$).
1. Using native python
2. Using numpy (precompiled functions in C)
3. Using native julia
4. Using native Julia with macros

In [ ]:
# using Pkg
# Pkg.activate(".")
# Pkg.add(["BenchmarkTools", "PythonCall"])
# Pkg.instantiate()

using BenchmarkTools # Package for evaluating execution speed
using PythonCall # Package that allows to call Python from Julia (loads numpy library as specified in CondaPkg.toml)

np = pyimport("numpy");

In [ ]:
# Native Python Function (Runs in Python's bytecode interpreter)
@pyexec """
def euclidean_native_py(x, y):
    total = 0.0
    for i in range(len(x)):
        diff = x[i] - y[i]
        total += diff * diff
    return total ** 0.5
""" => euclidean_native_py

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl(x, y)
    total = 0.0
    for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end

# Native Julia Function (Plain loop, JIT-compiled to native CPU machine code)
function euclidean_native_jl_fast(x, y)
    total = 0.0
    # @ is a macro that optimizes the compilation of the loop
    # @inbound tells the compiler to skip bounds checking for array indexing, which can improve performance
    # @simd, Single Instruction, Multiple Data, lets the compiler parallelize the operation
    @inbounds @simd for i in eachindex(x, y)
        diff = x[i] - y[i]
        total += diff * diff
    end
    return sqrt(total)
end;

In [ ]:
# create random array of length N with Float64 values between 0 and 1
N = 10_000_000
a_jl = rand(Float64, N)
b_jl = rand(Float64, N)

# Convert data into Python list and NumPy array formats
a_pylist = pylist(a_jl);       b_pylist = pylist(b_jl);
a_numpy  = np.array(a_jl);     b_numpy  = np.array(b_jl);

# @btime is a macro from BenchmarkTools.jl that measures the execution time of a function call
println("1. Native Python (Interpreted for-loop):")
@btime $euclidean_native_py($a_pylist, $b_pylist)

println("\n2. Python + NumPy (Pre-compiled C library):")
@btime $np.linalg.norm($a_numpy - $b_numpy)

println("\n3. Native Julia (Pure Julia for-loop, JIT compiled):")
@btime euclidean_native_jl($a_jl, $b_jl)

println("\n4. Native Julia (Optimized for-loop, JIT compiled):")
@btime euclidean_native_jl_fast($a_jl, $b_jl);

**Things to Notice**
1. **Native Python is slowest**: As expected, the pure Python loop takes the longest because the CPython interpreter has to check variable types and create new float objects in memory on every single iteration.
2. **Native Julia is even faster than NumPy**: even though NumPy also used pre-compiled C code, it is slower than Julia, becuase it evaluates the function `np.linalg.norm(a-b)` in two steps. First it runs the $a-b$ statement and writes it temporarily into RAM (9 allocations), then it runs the NumPy function. This extra step is what makes it slower than Julia which executes everything as a single pass without allocating temporary arrays. 
3. **@inbound and @simd**: both optimization speed up the Julia implementation compared to the basic Julia implementation without macros.
4. **Warm-Up**: The very first time you run a Julia function, there is a brief pause while Julia's JIT compiler translates the code into native machine instructions. The @btime macro excludes this one-time compilation overhead by executing an untimed warm-up run first and then reporting the minimum execution time across multiple runs.

### **CMIP Data Analysis**: Climate Stripes
**Packages**
* **Rasters**: Julia package for labeled multidimensional array and geospatial raster manipulation.
* **NCDatasets**: Reading, writing, and inspecting NetCDF files.
* **Dates**: Handling calendar dates, timestamps, and time intervals.
* **Statistics**: Statistical operations like `mean` and `std`.
* **CairoMakie**: Visualizing data.
* **GeoMakie**: Cartographic extension for Makie.

In [ ]:
# Pkg.add(["Rasters", "NCDatasets", "Dates", "Statistics"])
# Pkg.add(["CairoMakie", "GeoMakie"])

using Rasters, Dates, Statistics, NCDatasets
using CairoMakie, GeoMakie

#### Load Data

In [ ]:
intake_esgf = pyimport("intake_esgf")

# Search and download (reuses the files in ~/.esgf if already there)
catalog = intake_esgf.ESGFCatalog()     # don't call this `cat`: it would hide Julia's cat()
catalog.search(
    project       = "CMIP6",
    experiment_id = pylist(["historical", "ssp585"]),
    source_id     = "ACCESS-CM2",
    variable_id   = "tas",
    frequency     = "mon",
    variant_label = "r1i1p1f1",
    file_end      = "2100-12",
)
paths = catalog.to_path_dict()

# Convert to a Julia Dict: "historical" => [file, ...], "ssp585" => [file, ...]
# (Python keys look like "CMIP.historical", so keep only the part after the dot)
files = Dict(
    pyconvert(String, k.split(".")[-1]) => [pyconvert(String, pystr(p)) for p in v]
    for (k, v) in paths.items())

In [ ]:
# Open the files of each experiment (joining them along time in case a run is split into several files)
hist = cat([Raster(f, name=:tas) for f in files["historical"]]...; dims=Ti)
ssp  = cat([Raster(f, name=:tas) for f in files["ssp585"]]...; dims=Ti)

# Merge
hist   = modify(Array{Float32}, hist)
ssp    = modify(Array{Float32}, ssp[Ti=Where(t -> year(t) <= 2025)])
tas_raw = cat(hist, ssp; dims=Ti)

#### Calculate Temperature Anomaly

In [ ]:
tas = tas_raw .- 273.15f0
    
# Slice periods using precise end dates to ensure perfect 12-month multiples
tas_ref = tas[Ti(Where(x -> 1981 <= Dates.year(x) <= 2010))]
tas_anom = tas[Ti(Where(x -> 1979 <= Dates.year(x) <= 2025))]

nx, ny, nt_ref = size(tas_ref) # dimensions of reference period
nt_anom = size(tas_anom, 3) # time dimension of analysis period

# Reshape the 3D reference array into 4D: (Lon, Lat, Month, Year)
ref_reshaped = reshape(tas_ref.data, nx, ny, 12, nt_ref ÷ 12) # ÷ use for integer division

# Take mean across the year dimension
clim_data = mean(ref_reshaped, dims=4)

# Reshape the analysis array to match the 4D structure
anom_reshaped = reshape(tas_anom.data, nx, ny, 12, nt_anom ÷ 12)

# Subtract the monthly climatology
anom_data = anom_reshaped .- clim_data; # dim (192, 144, 12, 36) .- (192, 144, 12, 1)

# Convert back into raster file
anom_data = Raster(reshape(anom_data, nx, ny, nt_anom), dims(tas_anom))
mean_anom = mean(anom_data, dims=Ti)

#### Visualization Average Anomaly in Space

In [ ]:
# Remove dimensions for plotting
data_matrix = dropdims(mean_anom, dims=Ti)

# Extract coordinates and data and convert into native julia arrays
lons = collect(lookup(tas_anom, X))
lats = collect(lookup(tas_anom, Y))

# Shift longitudes from [0, 360] to [-180, 180] for GeoMakie Robinson projection
split_idx = findfirst(x -> x > 180, lons)

# Shift coordinates
lons_shifted = vcat(lons[split_idx:end] .- 360, lons[1:split_idx-1]) # vertical stack -180 : + 180
# Apply shift to the dataset
data_shifted = vcat(data_matrix[split_idx:end, :], data_matrix[1:split_idx-1, :])

In [ ]:
# Create Figure with standard Greenwich-centered projection
fig = Figure(size = (800, 500))
ga = GeoAxis( # create geographic coordinate system
    fig[1, 1];
    dest = "+proj=robin", # Robinson projection (Edges are now strictly ±180°) 
    title = "Near-Surface Air Temperature Anomalies (1979-2025)",
    titlesize = 16,
    xgridvisible = false, # hide grid lines
    ygridvisible = false
)
# Plot the surface using the shifted longitudes and data
sp = surface!(
    ga, # projection from Geoaxis
    lons_shifted, 
    lats, 
    data_shifted;
    colormap = :coolwarm,
    colorrange = (-0.5, 0.5),
    shading = NoShading
)

# add coastlines
lines!(ga, GeoMakie.coastlines(), color = :black, linewidth = 0.7)

# Add horizontal colorbar
Colorbar(
    fig[2, 1], sp; 
    vertical = false, 
    label = "Temperature anomaly (°C)", 
    width = Relative(0.7)
)

fig

#### Visualization Polar Amplification

In [ ]:
# Plot temperature curve of arctiv region
# Select 65-90°N and average over lon (X) and lat (Y)
tas_65_90N = anom_data[Y(65 .. 90)]
tas_65_90N_mean = mean(tas_65_90N, dims=(X, Y))
tas_65_90N_mean = dropdims(tas_65_90N_mean, dims=(X, Y))
tas_30_60N_mean = mean(anom_data[Y(30 .. 65)], dims=(X, Y))
tas_30_60N_mean = dropdims(tas_30_60N_mean, dims=(X, Y))

time_data = collect(lookup(tas_65_90N_mean, Ti)); # Extract a standard array from the Raster data
temp_data_65_90N = collect(tas_65_90N_mean); # Extract a standard array from the Raster data
temp_data_30_60N = collect(tas_30_60N_mean); # Extract a standard array from the Raster data

In [ ]:
fig = Figure(size = (1200, 400)) # create figure canvas

# Create the first Axis in row 1, column 1 (Left)
ax1 = Axis( 
    fig[1, 1], 
    xlabel = "Time",
    ylabel = "Temperature anomaly (°C)",
    title = "65–90°N"
)

# Create the second Axis in row 1, column 2 (Right)
ax2 = Axis(
    fig[1, 2], 
    xlabel = "Time",
    title = "30–60°N"
)

# Plot data into the first axis
lines!(ax1, time_data, temp_data_65_90N, color = :blue, linewidth = 1.5)
hlines!(ax1, [0.0], color = :black, linewidth = 0.8)

# Plot data into the second axis
lines!(ax2, time_data, temp_data_30_60N, color = :red, linewidth = 1.5)
hlines!(ax2, [0.0], color = :black, linewidth = 0.8)

linkyaxes!(ax1, ax2) # sync the y-axis of all plots
display(fig);

##### Visualize Climate Stripes

In [ ]:
temp_grouped = groupby(tas_65_90N, Ti => Dates.year)

In [ ]:
temperature = mean.(temp_grouped)

In [ ]:
years = collect(lookup(temperature, Ti)) # Extract the numeric years directly from the grouped Raster
temp_values = collect(temperature) # Extract the numeric temperature values

# Get the maximum absolute anomaly for a centered color scale
limit = maximum(abs.(temp_values))

fig = Figure(size = (1000, 400))
ax = Axis(fig[1, 1])

# Remove standard plot elements
hidedecorations!(ax) # remove x and y labels
hidespines!(ax)  # remove x and y axis lines

heatmap!(
    ax, 
    years, 
    [1], # y-axis dummy value
    reshape(temp_values, :, 1); # Reshape 1D vector to an (N_years, 1) matrix
    colormap = :bwr,
    colorrange = (-limit, limit)
)

display(fig);